#08 - Gold Publish
# Enterprise BFSI Client MDM Platform


###Purpose:

1. Final Gold certification boundary.
2. No duplicate Gold dataset is created.

###Setup

In [0]:

from pyspark.sql import functions as F
import uuid

CATALOG = "bfsi_mdm_dev"

EXPECTED_ATTRIBUTE_COUNT = 13

EXPECTED_GOLD_TABLES = [
    "master_client",
    "master_client_attribute",
    "client_source_map",
    "survivorship_audit",
    "master_client_history",
]

print("Notebook 08 initialized.")

Notebook 08 initialized.


###Verify Gold objects

In [0]:
# ============================================================
# VERIFY REQUIRED GOLD TABLES
# Standard-access-mode compatible
# ============================================================

existing_gold_tables = {
    row["tableName"]
    for row in (
        spark.sql(
            f"SHOW TABLES IN {CATALOG}.gold"
        )
        .select("tableName")
        .collect()
    )
}

for table_name in EXPECTED_GOLD_TABLES:
    assert table_name in existing_gold_tables, (
        f"Missing Gold table: {table_name}"
    )

print("All required Gold tables exist: PASS")
print(
    "Gold tables:",
    sorted(existing_gold_tables)
)

All required Gold tables exist: PASS
Gold tables: ['client_source_map', 'master_client', 'master_client_attribute', 'master_client_history', 'survivorship_audit']


###Run the complete Gold certification

In [0]:
master_client = spark.table(
    f"{CATALOG}.gold.master_client"
)

master_client_attribute = spark.table(
    f"{CATALOG}.gold.master_client_attribute"
)

source_map = spark.table(
    f"{CATALOG}.gold.client_source_map"
)

survivorship_audit = spark.table(
    f"{CATALOG}.gold.survivorship_audit"
)

history = spark.table(
    f"{CATALOG}.gold.master_client_history"
)

silver_count = spark.table(
    f"{CATALOG}.silver.all_clients"
).count()

# ------------------------------------------------------------
# Master Record
# ------------------------------------------------------------

master_rows = master_client.count()

duplicate_master_ids = (
    master_client
    .groupBy("master_client_id")
    .count()
    .filter(F.col("count") > 1)
)

assert (
    duplicate_master_ids
    .limit(1)
    .count()
    == 0
), (
    "Duplicate MASTER_CLIENT_ID values."
)

# ------------------------------------------------------------
# Source Map
# ------------------------------------------------------------

source_map_rows = source_map.count()

duplicate_source_mapping = (
    source_map
    .groupBy(
        "source_system",
        "source_id"
    )
    .count()
    .filter(F.col("count") > 1)
)

assert (
    duplicate_source_mapping
    .limit(1)
    .count()
    == 0
), (
    "One source record maps to multiple masters."
)

assert (
    source_map_rows == silver_count
), (
    "Source map does not cover Silver."
)

# ------------------------------------------------------------
# Provenance
# ------------------------------------------------------------

provenance_rows = (
    master_client_attribute.count()
)

duplicate_provenance = (
    master_client_attribute
    .groupBy(
        "master_client_id",
        "attribute_name"
    )
    .count()
    .filter(F.col("count") > 1)
)

assert (
    duplicate_provenance
    .limit(1)
    .count()
    == 0
), (
    "Duplicate master/attribute provenance rows."
)

assert (
    provenance_rows
    ==
    master_rows * EXPECTED_ATTRIBUTE_COUNT
), (
    "Provenance row count is not master_count × 13."
)

# ------------------------------------------------------------
# Survivorship Audit
# ------------------------------------------------------------

audit_rows = survivorship_audit.count()

assert (
    audit_rows
    ==
    master_rows * EXPECTED_ATTRIBUTE_COUNT
), (
    "Survivorship audit is incomplete."
)

# ------------------------------------------------------------
# SCD2
# ------------------------------------------------------------

current_history = (
    history
    .filter(
        F.col("is_current") == True
    )
)

duplicate_current_history = (
    current_history
    .groupBy(
        "master_client_id",
        "field"
    )
    .count()
    .filter(F.col("count") > 1)
)

assert (
    duplicate_current_history
    .limit(1)
    .count()
    == 0
), (
    "Multiple current SCD2 rows for a "
    "master/field."
)

assert (
    current_history.count()
    ==
    master_rows * EXPECTED_ATTRIBUTE_COUNT
), (
    "SCD2 current state is incomplete."
)

# ------------------------------------------------------------
# Referential integrity
# ------------------------------------------------------------

master_ids = (
    master_client
    .select("master_client_id")
    .distinct()
)

missing_source_map_master = (
    source_map
    .select("master_client_id")
    .distinct()
    .join(
        master_ids,
        "master_client_id",
        "left_anti"
    )
)

missing_provenance_master = (
    master_client_attribute
    .select("master_client_id")
    .distinct()
    .join(
        master_ids,
        "master_client_id",
        "left_anti"
    )
)

assert (
    missing_source_map_master
    .limit(1)
    .count()
    == 0
)

assert (
    missing_provenance_master
    .limit(1)
    .count()
    == 0
)

print("=" * 70)
print("ALL GOLD CERTIFICATION CHECKS: PASS")
print("=" * 70)

ALL GOLD CERTIFICATION CHECKS: PASS


###Write publication audit

In [0]:
run_id = str(
    uuid.uuid4()
)

gold_publish_audit = (
    spark.createDataFrame(
        [
            (
                run_id,
                master_rows,
                source_map_rows,
                provenance_rows,
                audit_rows,
                history.count(),
                current_history.count(),
                "SUCCESS",
                "ALL_GOLD_VALIDATIONS_PASSED",
            )
        ],
        [
            "run_id",
            "master_client_rows",
            "source_map_rows",
            "provenance_rows",
            "survivorship_audit_rows",
            "history_rows",
            "current_history_rows",
            "status",
            "validation_status",
        ]
    )
    .withColumn(
        "published_at",
        F.current_timestamp()
    )
)

(
    gold_publish_audit
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{CATALOG}.audit.pipeline_run_audit"
    )
)

print(
    f"Publication audit written. run_id={run_id}"
)

Publication audit written. run_id=ea0c6e1c-0b53-4464-ac49-e27105634755


###Final certification summary

In [0]:
print("=" * 80)
print("GOLD PUBLICATION COMPLETE")
print("=" * 80)

print(
    f"MASTER_CLIENT rows       : "
    f"{master_rows:,}"
)

print(
    f"CLIENT_SOURCE_MAP rows   : "
    f"{source_map_rows:,}"
)

print(
    f"PROVENANCE rows          : "
    f"{provenance_rows:,}"
)

print(
    f"SURVIVORSHIP AUDIT rows  : "
    f"{audit_rows:,}"
)

print(
    f"SCD2 history rows        : "
    f"{history.count():,}"
)

print(
    f"SCD2 current rows        : "
    f"{current_history.count():,}"
)

print(
    f"Publication run ID       : "
    f"{run_id}"
)

print(
    "STATUS                   : SUCCESS"
)

print("=" * 80)

GOLD PUBLICATION COMPLETE
MASTER_CLIENT rows       : 1,080,592
CLIENT_SOURCE_MAP rows   : 2,638,869
PROVENANCE rows          : 14,047,696
SURVIVORSHIP AUDIT rows  : 14,047,696
SCD2 history rows        : 14,047,696
SCD2 current rows        : 14,047,696
Publication run ID       : ea0c6e1c-0b53-4464-ac49-e27105634755
STATUS                   : SUCCESS
